# Iris Flower Classification

**Objective:** Train machine learning classification models to identify an iris flower species (`Setosa`, `Versicolor`, or `Virginica`) from its physical measurements.

**Dataset:** Built-in Iris dataset from `sklearn.datasets.load_iris()` — no external download required.

**Models covered:** Logistic Regression, K-Nearest Neighbours, Decision Tree, and Random Forest.

## 1. Import Libraries

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")

## 2. Load the Iris Dataset

In [ ]:
# Load the built-in Iris dataset from scikit-learn.
iris = load_iris()

# Convert the feature matrix into a pandas DataFrame for easier EDA.
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)

df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset shape: rows and columns.
df.shape

In [ ]:
# Data types for every column.
df.dtypes

In [ ]:
# Null value check.
df.isnull().sum()

In [ ]:
# Descriptive statistics for numerical features.
df.describe()

In [ ]:
# Class distribution.
df["species"].value_counts()

## 4. Visualisations

The pairplot shows relationships between every pair of features, colored by species. This is useful for spotting separability between classes.

In [ ]:
# Pairplot showing feature distributions and relationships by species.
sns.pairplot(df, hue="species", diag_kind="hist", corner=True)
plt.suptitle("Iris Feature Relationships by Species", y=1.02)
plt.show()

Box plots make it easier to compare each feature's distribution across the three species and identify discriminative measurements.

In [ ]:
# Box plots for each feature grouped by species.
feature_columns = iris.feature_names

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for axis, feature in zip(axes, feature_columns):
    sns.boxplot(data=df, x="species", y=feature, ax=axis)
    axis.set_title(f"{feature.title()} by Species")
    axis.set_xlabel("Species")

plt.tight_layout()
plt.show()

## 5. Feature Selection Discussion

From the visualisations, **petal length** and **petal width** appear to be the most discriminative features. `Setosa` is clearly separated from the other two species using petal measurements. `Versicolor` and `Virginica` overlap more, but petal length and petal width still separate them better than sepal length and sepal width.

For this baseline classification task, all four features are retained because the dataset is small, clean, and all measurements contain useful information. If we wanted a simpler model, petal length and petal width would be strong first choices.

## 6. Train/Test Split

In [ ]:
# Separate features and target.
X = df[feature_columns]
y = df["species"]

# Use an 80/20 split and stratify to preserve class proportions in train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

## 7. Train Multiple Classification Models

In [ ]:
# Define several classifiers for comparison.
models = {
    "Logistic Regression": LogisticRegression(max_iter=200, random_state=42),
    "K-Nearest Neighbours": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

results = []
predictions = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[model_name] = y_pred
    results.append({"Model": model_name, "Accuracy": accuracy_score(y_test, y_pred)})

results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)
results_df

## 8. Model Evaluation

Each model is evaluated using accuracy, confusion matrix, and a classification report containing precision, recall, and F1-score.

In [ ]:
for model_name, y_pred in predictions.items():
    print(f"\n{'=' * 70}")
    print(f"{model_name}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred, labels=iris.target_names))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

In [ ]:
# Visual confusion matrices for easier comparison.
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for axis, (model_name, y_pred) in zip(axes, predictions.items()):
    matrix = confusion_matrix(y_test, y_pred, labels=iris.target_names)
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=iris.target_names,
        yticklabels=iris.target_names,
        ax=axis,
    )
    axis.set_title(model_name)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")

plt.tight_layout()
plt.show()

## 9. Best-Performing Model

In [ ]:
# Identify the best model by highest test accuracy.
# If multiple models tie, all tied models are shown.
best_accuracy = results_df["Accuracy"].max()
best_models = results_df[results_df["Accuracy"] == best_accuracy]

print("Best model(s):")
display(best_models)

print(
    f"The best-performing model(s) achieved an accuracy of {best_accuracy:.4f}. "
    "The final choice should prefer the simplest tied model when performance is equal, "
    "because simpler models are easier to explain and maintain."
)

### Final Justification

The best-performing model is selected using test-set accuracy, supported by the confusion matrix and classification report. If two or more models tie, the simpler model is preferred. On the Iris dataset, simple models such as Logistic Regression and K-Nearest Neighbours often perform extremely well because the classes are largely separable, especially using petal length and petal width.

## References

- scikit-learn Iris dataset: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html
- scikit-learn supervised learning guide: https://scikit-learn.org/stable/supervised_learning.html
- seaborn pairplot documentation: https://seaborn.pydata.org/generated/seaborn.pairplot.html